# Two-group pySCENIC: Epi_VIM versus Epi_JUN

Run the cells in order. The workflow prepares two-group loom inputs, runs pySCENIC, and retains only the per-cell MWU CSV and ranked PDF from the final comparison step.


## 1. Prepare filtered two-group loom inputs


In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import loompy
import numpy as np
import pandas as pd
import scipy.sparse as sp


ADATA_QC_PATH = Path("../adata_qc.h5ad")
ADATA_EPI_PATH = Path("../adata_epi.h5ad")
OUT_DIR = Path("inputs")

GROUPS = ["Epi_VIM", "Epi_JUN"]
MAX_CELLS_PER_GROUP_FOR_GRN = 2000
RANDOM_SEED = 42


def get_two_group_obs() -> pd.DataFrame:
    adata_epi = ad.read_h5ad(ADATA_EPI_PATH, backed="r")
    obs = adata_epi.obs[["sample", "cell_subtype"]].copy()
    obs["sample"] = obs["sample"].astype(str)
    obs["cell_subtype"] = obs["cell_subtype"].astype(str)
    obs = obs[obs["cell_subtype"].isin(GROUPS)].copy()
    adata_epi.file.close()
    if obs.empty:
        raise ValueError(f"No cells found for {GROUPS}")
    missing = sorted(set(GROUPS) - set(obs["cell_subtype"]))
    if missing:
        raise ValueError(f"Missing groups in adata_epi: {missing}")
    return obs


def filter_genes_from_two_groups(obs: pd.DataFrame) -> tuple[list[str], pd.DataFrame]:
    adata_qc = ad.read_h5ad(ADATA_QC_PATH, backed="r")
    names = obs.index.intersection(adata_qc.obs_names)
    if len(names) != len(obs):
        raise ValueError(f"Only matched {len(names)}/{len(obs)} two-group cells in adata_qc")
    adata_sub = adata_qc[names, :].to_memory()
    adata_qc.file.close()

    x = adata_sub.X
    if not sp.issparse(x):
        x = sp.csr_matrix(x)
    else:
        x = x.tocsr()

    n_cells = x.shape[0]
    min_cells = int(np.ceil(0.01 * n_cells))
    min_counts = int(np.ceil(3 * 0.01 * n_cells))
    detected_cells = np.asarray((x > 0).sum(axis=0)).ravel()
    total_counts = np.asarray(x.sum(axis=0)).ravel()
    keep = (detected_cells >= min_cells) & (total_counts >= min_counts)

    genes = np.asarray(adata_sub.var_names.astype(str))
    table = pd.DataFrame(
        {
            "gene": genes,
            "detected_cells": detected_cells,
            "total_counts": total_counts,
            "kept": keep,
            "min_cells": min_cells,
            "min_counts": min_counts,
            "n_cells_for_filter": n_cells,
        }
    )
    return genes[keep].tolist(), table


def write_loom(cell_names: pd.Index, obs: pd.DataFrame, genes: list[str], out_loom: Path) -> None:
    adata_qc = ad.read_h5ad(ADATA_QC_PATH, backed="r")
    adata_sub = adata_qc[cell_names, genes].to_memory()
    adata_qc.file.close()

    x = adata_sub.X
    if sp.issparse(x):
        x = x.toarray()
    else:
        x = np.asarray(x)

    row_attrs = {"Gene": np.asarray(adata_sub.var_names.astype(str))}
    col_attrs = {
        "CellID": np.asarray(adata_sub.obs_names.astype(str)),
        "sample": np.asarray(obs.loc[cell_names, "sample"].astype(str)),
        "cell_subtype": np.asarray(obs.loc[cell_names, "cell_subtype"].astype(str)),
    }
    loompy.create(str(out_loom), x.T, row_attrs=row_attrs, col_attrs=col_attrs)


def prepare_two_group_loom() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(RANDOM_SEED)

    obs = get_two_group_obs()
    genes, filter_table = filter_genes_from_two_groups(obs)
    filter_table.to_csv(OUT_DIR / "two_group_counts_filtered_genes.csv", index=False)

    full_cells = obs.index
    write_loom(
        full_cells,
        obs,
        genes,
        OUT_DIR / "two_group_counts_filtered_for_pyscenic_aucell.loom",
    )

    selected = []
    rows = []
    for group in GROUPS:
        names = obs.index[obs["cell_subtype"] == group].to_numpy()
        n_take = min(MAX_CELLS_PER_GROUP_FOR_GRN, len(names))
        chosen = rng.choice(names, size=n_take, replace=False)
        selected.extend(chosen.tolist())
        rows.append({"cell_subtype": group, "available_cells": len(names), "selected_cells": n_take})
    selected_cells = pd.Index(selected)
    pd.DataFrame(rows).to_csv(OUT_DIR / "two_group_downsampled_cells.csv", index=False)
    write_loom(
        selected_cells,
        obs,
        genes,
        OUT_DIR / "two_group_counts_filtered_downsampled_for_pyscenic_grn.loom",
    )

    print(f"groups: {', '.join(GROUPS)}")
    print(obs["cell_subtype"].value_counts().sort_index().to_string())
    print(f"kept genes: {len(genes)}/{len(filter_table)}")
    print("wrote full AUCell loom and downsampled GRN loom")


prepare_two_group_loom()


## 2. Run pySCENIC GRN, motif pruning, and AUCell


In [ ]:
%%bash
set -euo pipefail

PYSCENIC="/mnt/disk18t/lr_xcy/micromamba/envs/stereopy_mouse/bin/pyscenic"
N_WORKERS="${N_WORKERS:-40}"

TF_LIST="../resources/allTFs_hg38.txt"
DB_500="../resources/hg38_500bp_up_100bp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"
DB_10K="../resources/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather"
MOTIFS="../resources/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl"
GRN_LOOM="inputs/two_group_counts_filtered_downsampled_for_pyscenic_grn.loom"
AUCELL_LOOM="inputs/two_group_counts_filtered_for_pyscenic_aucell.loom"

mkdir -p outputs logs

for required in "$PYSCENIC" "$TF_LIST" "$DB_500" "$DB_10K" "$MOTIFS" "$GRN_LOOM" "$AUCELL_LOOM"; do
  if [[ ! -e "$required" ]]; then
    echo "Missing required file: $required" >&2
    exit 1
  fi
done

"$PYSCENIC" grn \
  "$GRN_LOOM" \
  "$TF_LIST" \
  -o outputs/adjacencies.tsv \
  --num_workers "$N_WORKERS" \
  2>&1 | tee logs/01_grn.log

"$PYSCENIC" ctx \
  outputs/adjacencies.tsv \
  "$DB_500" "$DB_10K" \
  --annotations_fname "$MOTIFS" \
  --expression_mtx_fname "$GRN_LOOM" \
  --mode dask_multiprocessing \
  --output outputs/regulons.csv \
  --num_workers "$N_WORKERS" \
  2>&1 | tee logs/02_ctx.log

"$PYSCENIC" aucell \
  "$AUCELL_LOOM" \
  outputs/regulons.csv \
  --output outputs/auc_mtx.loom \
  --num_workers "$N_WORKERS" \
  2>&1 | tee logs/03_aucell.log


## 3. Per-cell regulon comparison and retained ranked plot


In [ ]:
from __future__ import annotations

from pathlib import Path

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from adjustText import adjust_text
from scipy import stats


AUC_LOOM = Path("outputs/auc_mtx.loom")
OUT_ROOT = Path("outputs/pairwise_regulon_activity")
PLOT_ROOT = Path("outputs/visualizations")

GROUP_A = "Epi_VIM"
GROUP_B = "Epi_JUN"
PAIR_NAME = f"{GROUP_A}__vs__{GROUP_B}"

TOP_N_LABELS = 5


plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["svg.fonttype"] = "none"


def decode_array(values) -> np.ndarray:
    arr = np.asarray(values)
    if arr.dtype.kind in {"S", "O"}:
        return np.array([x.decode() if isinstance(x, bytes) else str(x) for x in arr])
    return arr.astype(str)


def bh_fdr(pvalues: np.ndarray) -> np.ndarray:
    p = np.asarray(pvalues, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    adjusted = ranked * n / (np.arange(n) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.clip(adjusted, 0, 1)
    tmp = np.empty_like(adjusted)
    tmp[order] = adjusted
    out[valid] = tmp
    return out


def read_auc_loom() -> tuple[pd.DataFrame, pd.DataFrame]:
    with h5py.File(AUC_LOOM, "r") as handle:
        cell_ids = decode_array(handle["col_attrs/CellID"][:])
        subtypes = decode_array(handle["col_attrs/cell_subtype"][:])
        samples = decode_array(handle["col_attrs/sample"][:])
        auc_struct = handle["col_attrs/RegulonsAUC"][:]

    regulons = list(auc_struct.dtype.names)
    auc = pd.DataFrame(
        {regulon: auc_struct[regulon] for regulon in regulons},
        index=pd.Index(cell_ids, name="cell_id"),
    )
    meta = pd.DataFrame(
        {
            "cell_id": cell_ids,
            "cell_subtype": subtypes,
            "sample": samples,
        },
        index=auc.index,
    )
    return auc, meta


def run_per_cell_stats(auc: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    pair_mask = meta["cell_subtype"].isin([GROUP_A, GROUP_B])
    pair_auc = auc.loc[pair_mask]
    pair_meta = meta.loc[pair_mask].copy()
    a_idx = pair_meta.index[pair_meta["cell_subtype"] == GROUP_A]
    b_idx = pair_meta.index[pair_meta["cell_subtype"] == GROUP_B]

    rows = []
    for regulon in auc.columns:
        a = pair_auc.loc[a_idx, regulon].to_numpy(dtype=float)
        b = pair_auc.loc[b_idx, regulon].to_numpy(dtype=float)
        try:
            stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
        except ValueError:
            stat, p = np.nan, np.nan
        rows.append(
            {
                "regulon": regulon,
                "group": GROUP_A,
                "reference": GROUP_B,
                "n_group_cells": len(a),
                "n_reference_cells": len(b),
                "mean_group": np.mean(a),
                "mean_reference": np.mean(b),
                "median_group": np.median(a),
                "median_reference": np.median(b),
                "mean_diff": np.mean(a) - np.mean(b),
                "median_diff": np.median(a) - np.median(b),
                "mannwhitney_u": stat,
                "p": p,
            }
        )

    per_cell = pd.DataFrame(rows)
    per_cell["fdr"] = bh_fdr(per_cell["p"].to_numpy())
    per_cell = per_cell.sort_values("mean_diff", ascending=False).reset_index(drop=True)
    return per_cell


def save_per_cell_results(per_cell: pd.DataFrame) -> Path:
    pair_dir = OUT_ROOT / PAIR_NAME
    pair_dir.mkdir(parents=True, exist_ok=True)
    output_csv = pair_dir / f"{PAIR_NAME}_pyscenic_auc_per_cell_mwu.csv"
    per_cell.to_csv(output_csv, index=False)
    return output_csv


def plot_ranked_difference(df: pd.DataFrame, out_prefix: Path, value_col: str, fdr_col: str, title: str) -> None:
    plot_df = df.sort_values(value_col, ascending=False).reset_index(drop=True).copy()
    plot_df["rank"] = np.arange(1, len(plot_df) + 1)
    plot_df[fdr_col] = pd.to_numeric(plot_df[fdr_col], errors="coerce")
    plot_df["neg_log10_fdr"] = -np.log10(plot_df[fdr_col].fillna(1.0).clip(lower=1e-300))
    sig = plot_df[fdr_col] < 0.05

    fig, ax = plt.subplots(figsize=(14 * 2 / 3, 5))
    ax.scatter(plot_df.loc[~sig, "rank"], plot_df.loc[~sig, value_col], s=35, c="#c9c9c9", alpha=0.7)
    if sig.any():
        sc = ax.scatter(
            plot_df.loc[sig, "rank"],
            plot_df.loc[sig, value_col],
            s=45,
            c=plot_df.loc[sig, "neg_log10_fdr"],
            cmap="Reds",
            edgecolor="black",
            linewidth=0.3,
        )
        cbar = fig.colorbar(sc, ax=ax)
    else:
        sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap="Reds")
        cbar = fig.colorbar(sm, ax=ax)
    cbar.set_label("-log10(FDR)")

    label_idx = sorted(set(list(range(min(TOP_N_LABELS, len(plot_df)))) + list(plot_df.tail(TOP_N_LABELS).index)))
    texts = []
    target_x = []
    target_y = []
    label_dirs = []
    x_span = float(plot_df["rank"].max() - plot_df["rank"].min())
    y_span = float(plot_df[value_col].max() - plot_df[value_col].min())
    y_offset_base = max(0.001, 0.055 * y_span)
    min_line_len = max(0.0008, 0.035 * y_span)
    for idx in label_idx:
        row = plot_df.loc[idx]
        x0 = float(row["rank"])
        y0 = float(row[value_col])
        target_x.append(x0)
        target_y.append(y0)
        is_positive_side = idx < TOP_N_LABELS
        side_rank = idx if is_positive_side else (len(plot_df) - 1 - idx)
        direction = 1 if side_rank % 2 == 0 else -1
        label_dirs.append(direction)
        y_offset = y_offset_base * (1.0 + 0.18 * side_rank)
        text_x = x0
        text_y = y0 + direction * y_offset
        texts.append(
            ax.text(
                text_x,
                text_y,
                row["regulon"],
                fontsize=8,
                ha="center",
                va="center",
                zorder=4,
            )
        )
    if texts:
        adjust_text(
            texts,
            x=np.array(target_x, dtype=float),
            y=np.array(target_y, dtype=float),
            expand_points=(1.6, 1.8),
            expand_text=(1.2, 1.4),
            force_points=(0.2, 0.4),
            force_text=(0.5, 0.8),
            ax=ax,
        )
        for text_obj, x0, y0, direction in zip(texts, target_x, target_y, label_dirs):
            x1, y1 = text_obj.get_position()
            if abs(x1 - x0) < 0.25 and abs(y1 - y0) < min_line_len:
                y1 = y0 + direction * min_line_len
                text_obj.set_position((x1, y1))
            ax.plot([x0, x1], [y0, y1], color="black", lw=0.65, alpha=0.85, zorder=3)


    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    ax.set_xlabel("Rank")
    ax.set_ylabel(value_col)
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    fig.tight_layout()
    out_prefix.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_prefix.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)


def analyze_epi_vim_vs_epi_jun() -> None:
    auc, meta = read_auc_loom()
    if GROUP_A not in set(meta["cell_subtype"]) or GROUP_B not in set(meta["cell_subtype"]):
        raise ValueError(f"Need both {GROUP_A} and {GROUP_B} in auc_mtx.loom cell_subtype")

    per_cell = run_per_cell_stats(auc, meta)
    output_csv = save_per_cell_results(per_cell)

    pair_plot_dir = PLOT_ROOT / PAIR_NAME
    per_cell_rank = pair_plot_dir / f"{PAIR_NAME}_per_cell_mean_diff_ranked"
    plot_ranked_difference(
        per_cell,
        per_cell_rank,
        value_col="mean_diff",
        fdr_col="fdr",
        title=f"pySCENIC regulon AUC difference: {GROUP_A} vs {GROUP_B} (per cell MWU)",
    )

    print(f"Finished {PAIR_NAME}")
    print(f"per-cell results: {output_csv}")
    print(f"plot: {per_cell_rank.with_suffix('.pdf')}")


analyze_epi_vim_vs_epi_jun()
